# Usage of PySpark SQL

In [2]:
!pip install pyspark

     ---------------------------------------- 0.0/455.4 MB ? eta -:--:--
     --------------------------------------- 0.4/455.4 MB 12.2 MB/s eta 0:00:38
     --------------------------------------- 0.8/455.4 MB 10.1 MB/s eta 0:00:45
     --------------------------------------- 1.3/455.4 MB 10.6 MB/s eta 0:00:43
     --------------------------------------- 1.9/455.4 MB 10.8 MB/s eta 0:00:42
     --------------------------------------- 2.4/455.4 MB 10.9 MB/s eta 0:00:42
     --------------------------------------- 2.9/455.4 MB 10.8 MB/s eta 0:00:42
     --------------------------------------- 3.5/455.4 MB 11.3 MB/s eta 0:00:41
     --------------------------------------- 4.2/455.4 MB 11.7 MB/s eta 0:00:39
     --------------------------------------- 4.8/455.4 MB 11.9 MB/s eta 0:00:39
     --------------------------------------- 5.5/455.4 MB 12.1 MB/s eta 0:00:38
      -------------------------------------- 6.2/455.4 MB 12.4 MB/s eta 0:00:37
      -------------------------------------- 7.

  DEPRECATION: pyspark is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\monaa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


In [4]:
spark

In [5]:
sc = spark.sparkContext

In [6]:
## documentation
spark.read??

Type:        property
String form: <property object at 0x000002444CDA5D00>
Source:     
# spark.read.fget
@property
def read(self) -> DataFrameReader:
    """
    Returns a :class:`DataFrameReader` that can be used to read data
    in as a :class:`DataFrame`.

    .. versionadded:: 2.0.0

    .. versionchanged:: 3.4.0
        Supports Spark Connect.

    Returns
    -------
    :class:`DataFrameReader`

    Examples
    --------
    >>> spark.read
    <...DataFrameReader object ...>

    Write a DataFrame into a JSON file and read it back.

    >>> import tempfile
    >>> with tempfile.TemporaryDirectory(prefix="read") as d:
    ...     # Write a DataFrame into a JSON file
    ...     spark.createDataFrame(
    ...         [{"age": 100, "name": "Hyukjin Kwon"}]
    ...     ).write.mode("overwrite").format("json").save(d)
    ...
    ...     # Read the JSON file as a DataFrame.
    ...     spark.read.format('json').load(d).show()
    +---+------------+
    |age|        name|
    +---+--

In [7]:
file_path = "article.txt"

In [8]:
article = spark.read.text(file_path)

In [9]:
article

DataFrame[value: string]

In [10]:
article.printSchema()

root
 |-- value: string (nullable = true)



In [11]:
article.select(article.value)

DataFrame[value: string]

In [12]:
article.show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                       

In [13]:
from pyspark.sql.functions import col

In [14]:

article.select(article.value)
article.select(article['value'])
article.select(col('value'))
article.select('value')

DataFrame[value: string]

In [15]:
from pyspark.sql.functions import col, split

lines = article.select(
    split(col('value'), " ").alias('line')
)

In [16]:
lines.printSchema()

root
 |-- line: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [17]:
lines.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|line                                                                                                                                                                                                                                                             

In [18]:
lines

DataFrame[line: array<string>]

In [20]:
from pyspark.sql.functions import explode

words = lines.select(explode(col("line")).alias('word'))

In [21]:
words.printSchema()

root
 |-- word: string (nullable = false)



In [22]:
from pyspark.sql.functions import lower

words_lower = words.select(lower(col("word")).alias('word_lower'))

In [23]:
words_lower.show(10)

+----------+
|word_lower|
+----------+
|        if|
|       you|
|     spend|
|    enough|
|      time|
|    around|
|       the|
|      very|
|      rich|
|     these|
+----------+
only showing top 10 rows


In [24]:
from pyspark.sql.functions import regexp_extract

words_clean = words_lower.select(
    regexp_extract(col("word_lower"), r"(\W+)?([a-z]+)", 2).alias("word_clean")
)

In [25]:
words_clean.show(10)

+----------+
|word_clean|
+----------+
|        if|
|       you|
|     spend|
|    enough|
|      time|
|    around|
|       the|
|      very|
|      rich|
|     these|
+----------+
only showing top 10 rows


In [26]:
words_nonull = words_clean.where(col("word_clean") != "")

words_nonull.show(100)

+-----------+
| word_clean|
+-----------+
|         if|
|        you|
|      spend|
|     enough|
|       time|
|     around|
|        the|
|       very|
|       rich|
|      these|
|       days|
|         it|
|      clear|
|     people|
|       didn|
|        use|
|         to|
|       look|
|       like|
|       this|
|    because|
|     people|
|  naturally|
|        can|
|       look|
|       like|
|       this|
|     models|
|         in|
|          a|
|      paris|
|    fashion|
|       week|
|       show|
|        for|
|        the|
|     luxury|
|      brand|
|       mati|
|          f|
|       last|
|      month|
|caricatured|
|        the|
|    percent|
|         by|
|    wearing|
|prosthetics|
|       that|
|  resembled|
|       post|
|      faces|
|  including|
|  grotesque|
|      under|
|     bulges|
|       skin|
|     pulled|
|         up|
|       from|
|      their|
|    temples|
|        and|
|       lips|
|       that|
|   appeared|
|unnaturally|
|   inflated|
|     

In [27]:
groups = words_nonull.groupBy(col("word_clean"))

In [28]:
groups

GroupedData[grouping expressions: [word_clean], value: [word_clean: string], type: GroupBy]

In [29]:
counts = groups.count()

In [30]:
counts.orderBy('count', ascending=False).show(10)

+----------+-----+
|word_clean|count|
+----------+-----+
|       the|   45|
|        of|   31|
|       and|   28|
|         a|   28|
|        to|   25|
|        in|   17|
|      face|   15|
|     their|   10|
|        it|   10|
|      that|   10|
+----------+-----+
only showing top 10 rows


In [31]:
import pyspark.sql.functions as F

counts = (
    spark.read.text(file_path)
     .select(F.split(F.col('value'), ' ').alias('line'))
     .select(F.explode(F.col('line')).alias('word'))
     .select(F.lower(F.col('word')).alias('word'))
     .select(F.regexp_extract(F.col('word'), r"(\W+)?([a-z]+)", 2).alias('word'))
     .where(F.col('word') != "")
     .groupby('word')
     .count()
)

In [32]:
counts.show(10)

+----------+-----+
|      word|count|
+----------+-----+
|    brands|    2|
|      some|    1|
|     often|    3|
|   include|    1|
|variations|    1|
|  belonged|    1|
| reissuing|    1|
|  spending|    1|
|     ready|    1|
|      lips|    2|
+----------+-----+
only showing top 10 rows
